In [1]:
# 0. Libraries-ka loo baahan yahay

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# clustering model
from sklearn.cluster import KMeans

# scaling
from sklearn.preprocessing import StandardScaler

# evaluation metrics
from sklearn.metrics import silhouette_score, davies_bouldin_score

# si natiijooyinka mar kasta u degganaadaan
RANDOM_STATE = 42

In [2]:
# 1. Dataset-ka soo gelin

df = pd.read_csv("spending_l9_dataset.csv")

# eeg 5-ta row ee ugu horeysa
df.head()

,CustomerID,Age,Income_$,SpendingScore,VisitsPerMonth,OnlinePurchases,Gender,Region
0,1,28,33,78,14,9,Female,East
1,2,21,25,87,8,23,Male,North
2,3,23,24,88,13,10,Male,South
3,4,24,25,73,16,11,Female,West
4,5,20,23,88,17,16,Male,West


In [3]:
# dataset shape
print("Dataset shape:", df.shape)

# column names
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (200, 8)

Columns:
['CustomerID', 'Age', 'Income_$', 'SpendingScore', 'VisitsPerMonth', 'OnlinePurchases', 'Gender', 'Region']


In [4]:
# 2. Features-ka aan isticmaaleyno


# K-Means waxaan ka dooranay laba feature oo muhiim ah
selected_features = ["Income_$", "SpendingScore"]

X = df[selected_features].copy()

# haddii missing values jiraan, median ayaan ku buuxineynaa
for col in selected_features:
    if X[col].isna().any():
        X[col] = X[col].fillna(X[col].median())

# scaling waa muhiim sababtoo ah K-Means wuxuu ku shaqeeyaa distance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled data shape:", X_scaled.shape)

Scaled data shape: (200, 2)


In [5]:
# 
# 3. Elbow Method


sse_values = []

print("=== ELBOW METHOD RESULTS ===")
for k in range(1, 11):
    model = KMeans(n_clusters=k, n_init="auto", random_state=RANDOM_STATE)
    model.fit(X_scaled)

    sse_values.append(model.inertia_)
    print(f"k = {k} | SSE = {model.inertia_:.2f}")

=== ELBOW METHOD RESULTS ===
k = 1 | SSE = 400.00
k = 2 | SSE = 199.70
k = 3 | SSE = 79.37
k = 4 | SSE = 21.37
k = 5 | SSE = 19.09
k = 6 | SSE = 15.65
k = 7 | SSE = 14.48
k = 8 | SSE = 13.81
k = 9 | SSE = 12.94
k = 10 | SSE = 11.52


In [6]:
# 4. Final K-Means Model

# Elbow-ka kadib waxaan dooranay K = 5
best_k = 5

kmeans_model = KMeans(n_clusters=best_k, n_init="auto", random_state=RANDOM_STATE)

cluster_labels = kmeans_model.fit_predict(X_scaled)

# labels-ka dib ugu dar dataframe-ka
df["Cluster"] = cluster_labels.astype(int)

# eeg dhowr row
df.head()

,CustomerID,Age,Income_$,SpendingScore,VisitsPerMonth,OnlinePurchases,Gender,Region,Cluster
0,1,28,33,78,14,9,Female,East,2
1,2,21,25,87,8,23,Male,North,4
2,3,23,24,88,13,10,Male,South,4
3,4,24,25,73,16,11,Female,West,2
4,5,20,23,88,17,16,Male,West,4


In [7]:
# 5. Clustering Evaluation

sil_score = silhouette_score(X_scaled, cluster_labels)
dbi_score = davies_bouldin_score(X_scaled, cluster_labels)

print("=== CLUSTERING METRICS ===")
print(f"Silhouette Score      : {sil_score:.3f}")
print(f"Davies-Bouldin Index  : {dbi_score:.3f}")

=== CLUSTERING METRICS ===
Silhouette Score      : 0.642
Davies-Bouldin Index  : 0.571


In [8]:
# 6. Cluster Centers (Original Scale)

centers_scaled = kmeans_model.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=selected_features)
centers_df.index.name = "Cluster"

print("=== CLUSTER CENTERS ===")
print(centers_df.round(2))

=== CLUSTER CENTERS ===
         Income_$  SpendingScore
Cluster                         
0           56.32          53.58
1           28.92          19.60
2           25.33          78.04
3           99.16          79.24
4           22.74          89.04


In [9]:
# 7. Sanity Check

# saddex customer sample ah ayaan eegaynaa
sample_rows = [15, 62, 114]

sample_check = df.loc[sample_rows, selected_features + ["Cluster"]]

print("=== SANITY CHECK ===")
print(sample_check)

=== SANITY CHECK ===
     Income_$  SpendingScore  Cluster
15         19             86        4
62         68             51        0
114        43             14        1


In [10]:
# 8. Save Final Output

output_file = "spending_labeled_clusters.csv"

df.to_csv(output_file, index=False)

print(f"Final labeled dataset saved as: {output_file}")

Final labeled dataset saved as: spending_labeled_clusters.csv
